In [4]:
#1 Importing Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [5]:
#2 Checking Uploaded Files in Colab
print("Files in directory:")
print(os.listdir())

Files in directory:
['.config', 'customers.csv', 'data_dictionary.csv', 'complaints.csv', 'app_events.csv', 'deliveries.csv', 'hubs.csv', 'incidents.csv', 'drivers.csv', 'vehicles.csv', 'orders.csv', 'sample_data']


In [6]:
#3 Function to Automatically Load Files
def load_file(filename):
    files = os.listdir()
    for f in files:
        if filename.lower() in f.lower():
            print(f"Loading: {f}")
            return pd.read_csv(f)
    raise FileNotFoundError(f"{filename} not found!")

#4 Loading All Datasets
customers = load_file("customers")
orders = load_file("orders")
deliveries = load_file("deliveries")
drivers = load_file("drivers")
vehicles = load_file("vehicles")
hubs = load_file("hubs")
complaints = load_file("complaints")
incidents = load_file("incidents")
app_events = load_file("app_events")
data_dictionary = load_file("data_dictionary")

print("\nAll datasets loaded successfully!")

Loading: customers.csv
Loading: orders.csv
Loading: deliveries.csv
Loading: drivers.csv
Loading: vehicles.csv
Loading: hubs.csv
Loading: complaints.csv
Loading: incidents.csv
Loading: app_events.csv
Loading: data_dictionary.csv

All datasets loaded successfully!


In [7]:
#5 Storing Datasets in Dictionary
datasets = {
    "customers": customers,
    "orders": orders,
    "deliveries": deliveries,
    "drivers": drivers,
    "vehicles": vehicles,
    "hubs": hubs,
    "complaints": complaints,
    "incidents": incidents,
    "app_events": app_events
}

In [8]:
#6 Checking Dataset Shapes
print("\nDataset Shapes:")
for name, df in datasets.items():
    print(name, df.shape)


Dataset Shapes:
customers (650, 9)
orders (1250, 11)
deliveries (950, 13)
drivers (170, 8)
vehicles (120, 8)
hubs (8, 5)
complaints (320, 10)
incidents (280, 7)
app_events (640, 10)


In [9]:
#7 Previewing Sample Data
customers.head()

,customer_id,age,home_zone,customer_type,signup_date,loyalty_score,app_engagement_score,preferred_channel,account_status
0,C0001,26,North,SME,2024-11-27 04:25:00,44.9,69.2,App,Active
1,C0002,61,AIRPORT,Consumer,2025-10-28 01:04:00,55.4,66.6,App,Active
2,C0003,66,East,Consumer,2025-07-02 03:23:00,75.9,33.8,NaN,Active
3,C0004,75,CENTRAL,Consumer,2025-08-19 01:58:00,32.5,33.0,App,Active
4,C0005,26,Riverside,Consumer,2025-06-03 06:02:00,55.9,100.0,Web,Active


In [10]:
#8 Checking Missing Values
print("\nMissing Values:")
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())


Missing Values:

customers
customer_id              0
age                      0
home_zone                0
customer_type            0
signup_date              0
loyalty_score           20
app_engagement_score     0
preferred_channel       13
account_status           0
dtype: int64

orders
order_id                  0
customer_id               0
service_type              0
order_created_at          0
promised_window_hours     0
pickup_zone               0
dropoff_zone              0
priority_level            0
order_value               0
booking_channel          25
special_handling_flag     0
dtype: int64

deliveries
delivery_id                       0
order_id                          0
driver_id                         0
vehicle_id                        0
hub_id                            0
dispatch_time                     0
delivery_completed_at            19
delivery_status                   0
route_distance_km                 0
manual_route_override_count       0
proof_of_comple

In [11]:
#9 Checking Duplicate Records
print("\nDuplicate Rows:")
for name, df in datasets.items():
    print(name, df.duplicated().sum())


Duplicate Rows:
customers 0
orders 0
deliveries 0
drivers 0
vehicles 0
hubs 0
complaints 0
incidents 0
app_events 0


In [12]:
#10 Cleaning Text Columns
for name, df in datasets.items():
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace("nan", np.nan)

print("\nText columns cleaned successfully")


Text columns cleaned successfully


In [13]:
#11 Converting Date Columns
date_columns = [
    "signup_date", "order_created_at", "dispatch_time",
    "delivery_completed_at", "commission_date",
    "created_at", "reported_at", "event_timestamp"
]

for name, df in datasets.items():
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

print("\nDate columns converted successfully")


Date columns converted successfully


In [14]:
#12 Removing Duplicate Records
for name in datasets:
    datasets[name] = datasets[name].drop_duplicates()

print("\nDuplicate rows removed successfully")


Duplicate rows removed successfully


In [15]:
#13 Handling Missing Values
for name, df in datasets.items():
    for col in df.columns:
        if df[col].dtype in ["int64", "float64"]:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna("Unknown")

print("\nMissing values handled successfully")


Missing values handled successfully


In [16]:
#14 Feature Engineering

# Delivery Features
if "delivery_status" in deliveries.columns:
    deliveries["failed_delivery_flag"] = np.where(deliveries["delivery_status"] == "Failed", 1, 0)
    deliveries["delayed_delivery_flag"] = np.where(deliveries["delivery_status"] == "Delayed", 1, 0)

if "dispatch_time" in deliveries.columns and "delivery_completed_at" in deliveries.columns:
    deliveries["delivery_duration_hours"] = (
        deliveries["delivery_completed_at"] - deliveries["dispatch_time"]
    ).dt.total_seconds() / 3600

# Order Features
if "order_value" in orders.columns:
    orders["high_value_order"] = np.where(
        orders["order_value"] > orders["order_value"].median(), 1, 0
    )

# Complaint Features
if "status" in complaints.columns:
    complaints["unresolved_flag"] = np.where(
        complaints["status"].isin(["Open", "Escalated"]), 1, 0
    )

# App Event Features
if "success_flag" in app_events.columns:
    app_events["event_success_status"] = np.where(
        app_events["success_flag"] == 1, "Success", "Failed"
    )

print("\nFeature engineering completed")


Feature engineering completed


In [17]:
#15 Final Validation
print("\nFinal Check:")
for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Missing:", df.isnull().sum().sum())
    print("Duplicates:", df.duplicated().sum())


Final Check:

customers
Shape: (650, 9)
Missing: 0
Duplicates: 0

orders
Shape: (1250, 11)
Missing: 0
Duplicates: 0

deliveries
Shape: (950, 13)
Missing: 0
Duplicates: 0

drivers
Shape: (170, 8)
Missing: 0
Duplicates: 0

vehicles
Shape: (120, 8)
Missing: 0
Duplicates: 0

hubs
Shape: (8, 5)
Missing: 0
Duplicates: 0

complaints
Shape: (320, 10)
Missing: 0
Duplicates: 0

incidents
Shape: (280, 7)
Missing: 0
Duplicates: 0

app_events
Shape: (640, 10)
Missing: 0
Duplicates: 0


In [18]:
#16 Saving Cleaned Datasets
customers.to_csv("cleaned_customers.csv", index=False)
orders.to_csv("cleaned_orders.csv", index=False)
deliveries.to_csv("cleaned_deliveries.csv", index=False)
drivers.to_csv("cleaned_drivers.csv", index=False)
vehicles.to_csv("cleaned_vehicles.csv", index=False)
hubs.to_csv("cleaned_hubs.csv", index=False)
complaints.to_csv("cleaned_complaints.csv", index=False)
incidents.to_csv("cleaned_incidents.csv", index=False)
app_events.to_csv("cleaned_app_events.csv", index=False)

print("\nCleaned datasets saved successfully!")


Cleaned datasets saved successfully!


In [19]:
#17 Download Cleaned Datasets from Colab

from google.colab import files

files.download("cleaned_customers.csv")
files.download("cleaned_orders.csv")
files.download("cleaned_deliveries.csv")
files.download("cleaned_drivers.csv")
files.download("cleaned_vehicles.csv")
files.download("cleaned_hubs.csv")
files.download("cleaned_complaints.csv")
files.download("cleaned_incidents.csv")
files.download("cleaned_app_events.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>